# InstantID Generation Server

Runs an identity-conditioned image generation API on Kaggle T4 GPU and exposes it
via a cloudflared tunnel for use with `scripts/instantid_client.py`.

**Before running:**
- Accelerator: Settings → Accelerator → GPU T4 x1
- Internet: Settings → Internet → On

**After running all cells**, copy the printed `INSTANTID_SERVER_URL` into your `.env` file.
The tunnel stays alive as long as this notebook session is active (~9hr on Kaggle free tier).

In [ ]:
# Cell 1 — Install dependencies
import subprocess, sys

packages = [
    'insightface',
    'onnxruntime-gpu',
    'diffusers>=0.25.0',
    'accelerate',
    'transformers',
    'controlnet-aux',
    'flask',
    'huggingface_hub',
    'opencv-python-headless',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages, check=True)
print('Dependencies installed.')

In [ ]:
# Cell 2 — Download InstantID weights and custom pipeline
import os, shutil, subprocess
from huggingface_hub import hf_hub_download, snapshot_download

os.makedirs('./checkpoints', exist_ok=True)

# Clone InstantID repo for the custom diffusers pipeline script
if not os.path.exists('/tmp/InstantID'):
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/InstantX/InstantID', '/tmp/InstantID'],
        check=True
    )
shutil.copy('/tmp/InstantID/pipeline_stable_diffusion_xl_instantid.py', '.')
print('Custom pipeline script copied.')

# InstantID IP-Adapter weights
hf_hub_download(
    repo_id='InstantX/InstantID',
    filename='ip-adapter.bin',
    local_dir='./checkpoints',
)
print('ip-adapter.bin downloaded.')

# InstantID ControlNet weights (~2.5 GB)
snapshot_download(
    repo_id='InstantX/InstantID',
    allow_patterns=['ControlNetModel/*'],
    local_dir='./checkpoints',
)
print('ControlNet weights downloaded.')

In [ ]:
# Cell 3 — Download base SDXL model
# Uses stabilityai/stable-diffusion-xl-base-1.0 as the safe default (~6.9 GB).
# For better portrait fidelity, swap repo_id for e.g. 'SG161222/RealVisXL_V4.0'
# (same size, photorealism-tuned). Either works with InstantID.
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id='stabilityai/stable-diffusion-xl-base-1.0',
    local_dir='./checkpoints/base_model',
    ignore_patterns=['*.gguf', '*.fp32.*', 'vae/*'],
)
print('Base model downloaded.')

In [ ]:
# Cell 4 — Load InsightFace + InstantID pipeline
import sys
sys.path.insert(0, '.')

import torch
import cv2
import numpy as np
from PIL import Image
from insightface.app import FaceAnalysis
from diffusers.models import ControlNetModel
from diffusers import AutoencoderKL
from pipeline_stable_diffusion_xl_instantid import (
    StableDiffusionXLInstantIDPipeline,
    draw_kps,
)

print('Loading InsightFace (buffalo_l)...')
face_app = FaceAnalysis(
    name='buffalo_l',
    root='/tmp/insightface',
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider'],
)
face_app.prepare(ctx_id=0, det_size=(640, 640))
print('InsightFace loaded.')

print('Loading ControlNet...')
controlnet = ControlNetModel.from_pretrained(
    './checkpoints/ControlNetModel',
    torch_dtype=torch.float16,
)

# fp16-fixed VAE prevents NaN artifacts at half precision
print('Loading VAE...')
vae = AutoencoderKL.from_pretrained(
    'madebyollin/sdxl-vae-fp16-fix',
    torch_dtype=torch.float16,
)

print('Loading pipeline...')
pipe = StableDiffusionXLInstantIDPipeline.from_pretrained(
    './checkpoints/base_model',
    controlnet=controlnet,
    vae=vae,
    torch_dtype=torch.float16,
    safety_checker=None,
)
pipe.cuda()
pipe.load_ip_adapter_instantid('./checkpoints/ip-adapter.bin')
print('Pipeline ready.')

In [ ]:
# Cell 5 — Generation function + Flask API
import base64, io, threading
from flask import Flask, request, jsonify


def generate_images(
    face_image: Image.Image,
    positive_prompt: str,
    negative_prompt: str = '',
    num_images: int = 1,
    guidance_scale: float = 7.0,
    seed: int | None = None,
    ip_adapter_scale: float = 0.8,
    controlnet_scale: float = 0.8,
) -> list[Image.Image]:
    face_np = cv2.cvtColor(np.array(face_image), cv2.COLOR_RGB2BGR)
    faces = face_app.get(face_np)
    if not faces:
        raise ValueError('No face detected in reference image')
    # Use largest detected face
    face = sorted(faces, key=lambda f: (
        (f['bbox'][2] - f['bbox'][0]) * (f['bbox'][3] - f['bbox'][1])
    ))[-1]
    face_emb = face['embedding']
    face_kps = draw_kps(face_image, face['kps'])

    pipe.set_ip_adapter_scale(ip_adapter_scale)
    generator = (
        torch.Generator('cuda').manual_seed(seed)
        if seed is not None else None
    )
    result = pipe(
        prompt=positive_prompt,
        negative_prompt=negative_prompt,
        image_embeds=face_emb,
        image=face_kps,
        controlnet_conditioning_scale=controlnet_scale,
        num_inference_steps=30,
        guidance_scale=guidance_scale,
        num_images_per_prompt=num_images,
        generator=generator,
    )
    return result.images


flask_app = Flask(__name__)

@flask_app.route('/health')
def health():
    return jsonify({'status': 'ok', 'gpu': torch.cuda.is_available()})

@flask_app.route('/generate', methods=['POST'])
def generate_endpoint():
    data = request.json
    try:
        face_bytes = base64.b64decode(data['face_image_b64'])
        face_image = Image.open(io.BytesIO(face_bytes)).convert('RGB')
        images = generate_images(
            face_image=face_image,
            positive_prompt=data['positive_prompt'],
            negative_prompt=data.get('negative_prompt', ''),
            num_images=data.get('num_images', 1),
            guidance_scale=data.get('guidance_scale', 7.0),
            seed=data.get('seed'),
            ip_adapter_scale=data.get('ip_adapter_scale', 0.8),
            controlnet_scale=data.get('controlnet_scale', 0.8),
        )
        encoded = []
        for img in images:
            buf = io.BytesIO()
            img.save(buf, format='PNG')
            encoded.append(base64.b64encode(buf.getvalue()).decode())
        return jsonify({'images': encoded, 'count': len(encoded)})
    except Exception as exc:
        return jsonify({'error': str(exc)}), 500


server_thread = threading.Thread(
    target=lambda: flask_app.run(host='0.0.0.0', port=5000, debug=False)
)
server_thread.daemon = True
server_thread.start()
print('Flask server started on :5000')

In [ ]:
# Cell 6 — Cloudflared tunnel — run last, keep this cell alive
# The URL printed below is what goes in .env as INSTANTID_SERVER_URL.
# The tunnel lives as long as this Kaggle session is active.
import os, re, subprocess, time

cloudflared = '/tmp/cloudflared'
if not os.path.exists(cloudflared):
    subprocess.run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', cloudflared,
    ], check=True)
    os.chmod(cloudflared, 0o755)

proc = subprocess.Popen(
    [cloudflared, 'tunnel', '--url', 'http://localhost:5000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

tunnel_url = None
print('Waiting for tunnel...')
for line in proc.stdout:
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

print()
print('=' * 60)
print(f'INSTANTID_SERVER_URL={tunnel_url}')
print('Add the above line to your .env file on the GTA server.')
print('=' * 60)

# Keep session alive — the tunnel runs in proc
try:
    proc.wait()
except KeyboardInterrupt:
    proc.terminate()
    print('Tunnel closed.')